# Qwen3.5-0.8B × TinyCeNN PDelta3-CLVR

Sequentially replace only Qwen3.5 full-attention layers with PDelta3/GDN2 + Local32 + CLVR. The Colab uses device-safe launchers for training, verification, and Hugging Face release. Newly created recurrent cores are explicitly moved to the same device as the Qwen attention layer before the first forward pass.

In [1]:
import os, sys, pathlib, subprocess

os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','--upgrade',
    'transformers==5.17.0','datasets>=3,<5','huggingface_hub>=0.34,<2',
    'safetensors','pandas','matplotlib','-e',str(REPO_DIR)
], check=True)

preflight = (
    "import transformers; "
    "from transformers import Qwen3_5ForCausalLM; "
    "from transformers.models.qwen3_5.modeling_qwen3_5 import apply_rotary_pos_emb; "
    "print('transformers:', transformers.__version__); "
    "print('Qwen3.5 API preflight: OK')"
)
subprocess.run([sys.executable,'-c',preflight], check=True)

for p in (REPO_DIR/'src', REPO_DIR, REPO_DIR/'scripts'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
subprocess.run(['nvidia-smi'], check=False)

torch: 2.11.0+cu128 cuda: True


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/TinyCeNN-LM')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [3]:
BASE_MODEL = 'Qwen/Qwen3.5-0.8B'
FEATURE_DIM = 96
LOCAL_WINDOW = 32
CHUNK_SIZE = 32
CONV_KERNEL = 4
STATE_DTYPE = 'fp16'
LOCAL_GATE_INIT = 0.72
TARGET_FULL_LAYERS = 3
CONTEXT_LENGTH = 128
PROBE_CONTEXT = 128
PROBE_BLOCKS = 6
SEED = 2026
MIN_LAYER_STEPS = 60
MAX_LAYER_STEPS = 250
CHECK_EVERY = 25
LAYER_LR = 2e-4
QKV_LR_SCALE = 0.10
TEMPERATURE = 1.5
FUNCTIONAL_WEIGHT = 0.30
KL_WEIGHT = 1.00
CE_WEIGHT = 0.08
COSINE_WEIGHT = 0.20
LOCAL_GATE_PENALTY = 0.001
RESCUE_LR_SCALE = 0.50
RESCUE_FUNCTIONAL_WEIGHT = 0.15
RESCUE_KL_WEIGHT = 1.50
RESCUE_CE_WEIGHT = 0.12
ACCEPT_NMSE = 0.15
ACCEPT_COSINE = 0.94
ACCEPT_INCREMENTAL_DELTA_NLL = 0.015
ACCEPT_CUMULATIVE_DELTA_NLL = 0.05
RESUME = True
MAX_ROUNDS_PER_RUN = 2
MAX_RUNTIME_MINUTES = 240

OUTPUT_DIR = DRIVE_ROOT / f'qwen35-0.8b-pdelta3-gdn2-clvr-local{LOCAL_WINDOW}-f{FEATURE_DIM}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = 'vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32'
print('Output:', OUTPUT_DIR)
print('HF repo:', HF_REPO_ID)

Output: /content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-pdelta3-gdn2-clvr-local32-f96
HF repo: vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32


## Train / resume

This cell pulls the latest repository changes before every run. The device-safe launcher moves each newly created PDelta3 replacement to the Qwen layer device and verifies that no replacement parameters remain on CPU when the model is on CUDA.

In [4]:
import signal

subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)

cmd = [
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_pdelta3_clvr_sequential_colab.py'),
    '--base-model',BASE_MODEL,'--output-dir',str(OUTPUT_DIR),
    '--feature-dim',str(FEATURE_DIM),'--local-window',str(LOCAL_WINDOW),
    '--chunk-size',str(CHUNK_SIZE),'--conv-kernel',str(CONV_KERNEL),
    '--state-dtype',STATE_DTYPE,'--local-gate-init',str(LOCAL_GATE_INIT),
    '--target-full-layers',str(TARGET_FULL_LAYERS),'--context-length',str(CONTEXT_LENGTH),
    '--probe-context',str(PROBE_CONTEXT),'--probe-blocks',str(PROBE_BLOCKS),
    '--seed',str(SEED),'--min-layer-steps',str(MIN_LAYER_STEPS),
    '--max-layer-steps',str(MAX_LAYER_STEPS),'--check-every',str(CHECK_EVERY),
    '--layer-lr',str(LAYER_LR),'--qkv-lr-scale',str(QKV_LR_SCALE),
    '--temperature',str(TEMPERATURE),'--functional-weight',str(FUNCTIONAL_WEIGHT),
    '--kl-weight',str(KL_WEIGHT),'--ce-weight',str(CE_WEIGHT),
    '--cosine-weight',str(COSINE_WEIGHT),'--local-gate-penalty',str(LOCAL_GATE_PENALTY),
    '--rescue-lr-scale',str(RESCUE_LR_SCALE),
    '--rescue-functional-weight',str(RESCUE_FUNCTIONAL_WEIGHT),
    '--rescue-kl-weight',str(RESCUE_KL_WEIGHT),'--rescue-ce-weight',str(RESCUE_CE_WEIGHT),
    '--accept-nmse',str(ACCEPT_NMSE),'--accept-cosine',str(ACCEPT_COSINE),
    '--accept-incremental-delta-nll',str(ACCEPT_INCREMENTAL_DELTA_NLL),
    '--accept-cumulative-delta-nll',str(ACCEPT_CUMULATIVE_DELTA_NLL),
    '--max-runtime-minutes',str(MAX_RUNTIME_MINUTES),
    '--warm-start-previous-core','--train-qkv','--strict-acceptance',
    '--resume' if RESUME else '--no-resume'
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['SEQUENTIAL_MAX_ROUNDS_PER_RUN'] = str(MAX_ROUNDS_PER_RUN)
env['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
log_path = OUTPUT_DIR/'last_colab_run.log'
print(' '.join(cmd))
print('Log:', log_path)

interrupted = False
with log_path.open('w',encoding='utf-8') as log:
    p = subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    assert p.stdout is not None
    try:
        for line in p.stdout:
            print(line,end='')
            log.write(line)
            log.flush()
        rc = p.wait()
    except KeyboardInterrupt:
        interrupted = True
        p.send_signal(signal.SIGINT)
        try:
            rc = p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            p.terminate()
            rc = p.wait()

print('Process return code:', rc)
if rc != 0 and not interrupted:
    print('\n--- LAST 80 LOG LINES ---')
    lines = log_path.read_text(encoding='utf-8',errors='replace').splitlines()
    print('\n'.join(lines[-80:]))
    raise RuntimeError(f'Qwen3.5 trainer failed with return code {rc}. See {log_path}.')

/usr/bin/python3 -u /content/TinyCeNN-LM/scripts/run_qwen35_pdelta3_clvr_sequential_colab.py --base-model Qwen/Qwen3.5-0.8B --output-dir /content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-pdelta3-gdn2-clvr-local32-f96 --feature-dim 96 --local-window 32 --chunk-size 32 --conv-kernel 4 --state-dtype fp16 --local-gate-init 0.72 --target-full-layers 3 --context-length 128 --probe-context 128 --probe-blocks 6 --seed 2026 --min-layer-steps 60 --max-layer-steps 250 --check-every 25 --layer-lr 0.0002 --qkv-lr-scale 0.1 --temperature 1.5 --functional-weight 0.3 --kl-weight 1.0 --ce-weight 0.08 --cosine-weight 0.2 --local-gate-penalty 0.001 --rescue-lr-scale 0.5 --rescue-functional-weight 0.15 --rescue-kl-weight 1.5 --rescue-ce-weight 0.12 --accept-nmse 0.15 --accept-cosine 0.94 --accept-incremental-delta-nll 0.015 --accept-cumulative-delta-nll 0.05 --max-runtime-minutes 240 --warm-start-previous-core --train-qkv --strict-acceptance --resume
Log: /content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-pdelta

In [5]:
import json, pandas as pd
for name in ('qwen35_run_status.json','qwen35_progress.json','qwen35_in_progress.json'):
    p = OUTPUT_DIR/name
    if p.exists():
        print('\n###', name)
        print(p.read_text()[:12000])

p = OUTPUT_DIR/'qwen35_progress.json'
if p.exists():
    reports = json.loads(p.read_text()).get('reports',[])
    if reports:
        df = pd.DataFrame(reports)
        cols = [c for c in ['layer','round','step','accepted','nmse','cosine','incremental_delta_nll','cumulative_delta_nll','local_gate_mean'] if c in df.columns]
        display(df[cols].tail(30))


### qwen35_run_status.json
{
  "status": "target_full_attention_prefix_accepted",
  "architecture": "Qwen3.5-PDelta3-GDN2-CLVR+LocalW",
  "base_model": "Qwen/Qwen3.5-0.8B",
  "native_full_attention_layers": [
    3,
    7,
    11,
    15,
    19,
    23
  ],
  "target_full_attention_layers": [
    3,
    7,
    11
  ],
  "accepted_full_attention_layers": [
    3,
    7,
    11
  ],
  "teacher_probe_nll": 2.861148993174235,
  "final_probe_nll": 2.8818757136662803,
  "final_delta_nll": 0.02072672049204538,
  "config": {
    "feature_dim": 96,
    "local_window": 32,
    "chunk_size": 32,
    "conv_kernel": 4,
    "state_dtype": "fp16",
    "variant": "conv4_gdn2_clvr_f96",
    "local_gate_init": 0.72,
    "warm_start_previous_core": true
  },
  "elapsed_minutes": 2.422297354216666,
  "peak_vram_gib": 4.466190814971924
}

### qwen35_progress.json
{
  "format_version": 1,
  "architecture": "qwen3.5-pdelta3-gdn2-clvr-localw",
  "accepted_full_attention_layers": [
    3,
    7,
    11
  ],


,layer,round,step,accepted,nmse,cosine,incremental_delta_nll,cumulative_delta_nll,local_gate_mean
0,3,1,75,True,0.062081,0.974463,-0.003246,-0.003246,0.720681
1,7,1,150,True,0.038897,0.970457,0.013274,0.010028,0.721335
2,11,1,75,True,0.104179,0.941103,0.010699,0.020727,0.721432


## 1. Prompt + quality verification

Verification uses the same device-safe replacement handoff and writes `qwen35_verification.json`.

In [6]:
subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)
verify_cmd = [
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_release_colab.py'),'verify',
    '--base-model',BASE_MODEL,'--output-dir',str(OUTPUT_DIR),
    '--probe-context',str(PROBE_CONTEXT),'--probe-blocks',str(PROBE_BLOCKS),
    '--accept-nmse',str(ACCEPT_NMSE),'--accept-cosine',str(ACCEPT_COSINE),
    '--accept-incremental-delta-nll',str(ACCEPT_INCREMENTAL_DELTA_NLL),
    '--accept-cumulative-delta-nll',str(ACCEPT_CUMULATIVE_DELTA_NLL)
]
subprocess.run(verify_cmd, check=True)
verification = json.loads((OUTPUT_DIR/'qwen35_verification.json').read_text())
print('verified =', verification['verified'])
print('delta_nll =', verification['delta_nll'])

verified = True
delta_nll = 0.02072672049204538


## 2. Upload verified model to Hugging Face

Upload is blocked unless verification passed. Authentication is requested only here.

In [10]:
from huggingface_hub import get_token, login, HfApi
import os
import subprocess
import sys

# Get latest repo changes
subprocess.run(
    ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
    check=True,
)

# Login if necessary
hf_token = get_token()

if not hf_token:
    print("Please log in with a Hugging Face WRITE token.")
    login(add_to_git_credential=False)
    hf_token = get_token()

if not hf_token:
    raise RuntimeError("Hugging Face login failed: no token available.")

# Verify authentication before starting the upload
api = HfApi(token=hf_token)
user = api.whoami()
print("Authenticated as:", user["name"])

# Pass token explicitly to the subprocess
env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["HUGGING_FACE_HUB_TOKEN"] = hf_token

upload_cmd = [
    sys.executable,
    "-u",
    str(REPO_DIR / "scripts" / "run_qwen35_release_colab.py"),
    "upload",
    "--base-model",
    BASE_MODEL,
    "--output-dir",
    str(OUTPUT_DIR),
    "--repo-id",
    HF_REPO_ID,
]

print("Uploading to:", HF_REPO_ID)

subprocess.run(
    upload_cmd,
    check=True,
    env=env,
)

print("✅ Upload completed:", f"https://huggingface.co/{HF_REPO_ID}")

Authenticated as: vtava
Uploading to: vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32
✅ Upload completed: https://huggingface.co/vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32


In [11]:
# Compare sample prompts: BASE Qwen3.5 vs PDelta3-CLVR accepted model

import gc
import json
import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

# Make repo modules importable
for p in (REPO_DIR / "src", REPO_DIR, REPO_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from train_qwen35_pdelta3_clvr_sequential import (
    QwenPDelta3CLVRConfig,
    replace_full_attention_layers,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
    if device.type == "cuda"
    else torch.float32
)

PROGRESS_FILE = OUTPUT_DIR / "qwen35_progress.pt"

if not PROGRESS_FILE.exists():
    raise FileNotFoundError(
        f"No accepted checkpoint found: {PROGRESS_FILE}"
    )

progress = torch.load(
    PROGRESS_FILE,
    map_location="cpu",
    weights_only=False,
)

accepted_layers = [
    int(x)
    for x in progress["accepted_full_attention_layers"]
]

if not accepted_layers:
    raise RuntimeError("No accepted PDelta3-CLVR layers found.")

replacement_cfg = QwenPDelta3CLVRConfig.from_dict(
    progress["config"]
)

print("Accepted PDelta3-CLVR layers:", accepted_layers)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,
    token=False,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


prompts = [
    "The future of small language models is",
    "Artificial intelligence can help scientists by",
    "A good software architecture should",
    "The capital of Austria is",
    "Once upon a time, a small robot",
    "Machine learning systems become more efficient when",
    "The main advantage of recurrent neural networks is",
    "Vienna is known for",
]


@torch.no_grad()
def generate_examples(model, prompts):
    model.eval()
    results = {}

    for prompt in prompts:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        ).to(device)

        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        results[prompt] = tokenizer.decode(
            output[0],
            skip_special_tokens=True,
        )

    return results


# ------------------------------------------------------------------
# 1. BASE MODEL
# ------------------------------------------------------------------

print("\nLoading BASE model...")

base_model = Qwen3_5ForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    token=False,
    attn_implementation="eager",
).to(device)

base_model.config.use_cache = False

base_results = generate_examples(
    base_model,
    prompts,
)

del base_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------------
# 2. PDELTA3-CLVR MODEL
# ------------------------------------------------------------------

print("Loading PDelta3-CLVR model...")

new_model = Qwen3_5ForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    token=False,
    attn_implementation="eager",
).to(device)

new_model.config.use_cache = False

# Insert custom replacement layers
wrappers = replace_full_attention_layers(
    new_model,
    replacement_cfg,
    accepted_layers,
)

# Important: new modules may initially be created on CPU.
# Move them explicitly to the device of the model.
for wrapper in wrappers:
    wrapper.to(device=device)

# Load trained replacement parameters
incompatible = new_model.load_state_dict(
    progress["attention_state"],
    strict=False,
)

# Check that accepted layers were loaded correctly
prefixes = tuple(
    f"model.layers.{i}.self_attn."
    for i in accepted_layers
)

missing = [
    key
    for key in incompatible.missing_keys
    if key.startswith(prefixes)
]

if missing:
    raise RuntimeError(
        "Missing PDelta3 checkpoint weights:\n"
        + "\n".join(missing[:20])
    )

new_results = generate_examples(
    new_model,
    prompts,
)


# ------------------------------------------------------------------
# 3. SIDE-BY-SIDE OUTPUT
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print(
    "BASE Qwen3.5 vs "
    f"PDelta3-CLVR accepted layers {accepted_layers}"
)
print("=" * 110)

for prompt in prompts:

    print("\n" + "=" * 110)
    print("PROMPT:")
    print(prompt)

    print("\nBASELINE QWEN3.5:")
    print(base_results[prompt])

    print("\nPDELTA3-CLVR:")
    print(new_results[prompt])


# Optional: save comparison to Google Drive
comparison = [
    {
        "prompt": prompt,
        "baseline": base_results[prompt],
        "pdelta3_clvr": new_results[prompt],
    }
    for prompt in prompts
]

comparison_file = OUTPUT_DIR / "prompt_comparison.json"

comparison_file.write_text(
    json.dumps(
        {
            "base_model": BASE_MODEL,
            "accepted_layers": accepted_layers,
            "examples": comparison,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nSaved comparison to:")
print(comparison_file)

Accepted PDelta3-CLVR layers: [3, 7, 11]

Loading BASE model...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Loading PDelta3-CLVR model...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


BASE Qwen3.5 vs PDelta3-CLVR accepted layers [3, 7, 11]

PROMPT:
The future of small language models is

BASELINE QWEN3.5:
The future of small language models is not just about the technology, but also about the human side of the conversation.

In the last few years, small language models (SLMs) have become a game changer in the world of AI. They are fast, cheap, and capable of handling complex tasks. However, they are also not without their limitations.

PDELTA3-CLVR:
The future of small language models is not just about the technology, but also about the human side of the conversation.

In the last few years, small language models (SLMs) have been gaining traction in the tech industry. They are becoming increasingly popular for their ability to generate text, code, and other tasks. However, they are also facing challenges

PROMPT:
Artificial intelligence can help scientists by

BASELINE QWEN3.5:
Artificial intelligence can help scientists by providing a way to predict the future.

<

In [12]:
# Interactive chat comparison:
# BASE Qwen3.5-0.8B vs accepted PDelta3-CLVR model

import gc
import sys
import torch
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

# ------------------------------------------------------------------
# Setup
# ------------------------------------------------------------------

for p in (REPO_DIR / "src", REPO_DIR, REPO_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from train_qwen35_pdelta3_clvr_sequential import (
    QwenPDelta3CLVRConfig,
    replace_full_attention_layers,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
    if device.type == "cuda"
    else torch.float32
)

progress_path = OUTPUT_DIR / "qwen35_progress.pt"

if not progress_path.exists():
    raise FileNotFoundError(
        f"Accepted checkpoint not found: {progress_path}"
    )

progress = torch.load(
    progress_path,
    map_location="cpu",
    weights_only=False,
)

accepted_layers = [
    int(x)
    for x in progress["accepted_full_attention_layers"]
]

if not accepted_layers:
    raise RuntimeError("No accepted PDelta3-CLVR layers found.")

replacement_cfg = QwenPDelta3CLVRConfig.from_dict(
    progress["config"]
)

print("Accepted replacement layers:", accepted_layers)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=False,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


# ------------------------------------------------------------------
# Load BASE model
# ------------------------------------------------------------------

print("\nLoading original Qwen3.5...")

base_model = Qwen3_5ForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    token=False,
    attn_implementation="eager",
).to(device)

base_model.config.use_cache = False


# ------------------------------------------------------------------
# Load PDelta3-CLVR model
# ------------------------------------------------------------------

print("Loading PDelta3-CLVR model...")

new_model = Qwen3_5ForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    token=False,
    attn_implementation="eager",
).to(device)

new_model.config.use_cache = False

wrappers = replace_full_attention_layers(
    new_model,
    replacement_cfg,
    accepted_layers,
)

# New replacement modules can be created on CPU.
for wrapper in wrappers:
    wrapper.to(device=device)

incompatible = new_model.load_state_dict(
    progress["attention_state"],
    strict=False,
)

prefixes = tuple(
    f"model.layers.{i}.self_attn."
    for i in accepted_layers
)

missing = [
    key
    for key in incompatible.missing_keys
    if key.startswith(prefixes)
]

if missing:
    raise RuntimeError(
        "Missing accepted replacement weights:\n"
        + "\n".join(missing[:20])
    )

new_model.eval()
base_model.eval()


# ------------------------------------------------------------------
# Chat helpers
# ------------------------------------------------------------------

def build_chat_prompt(messages):
    """
    Uses Qwen chat template if available.
    Falls back to a simple User/Assistant format.
    """
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            pass

    text = ""
    for m in messages:
        role = m["role"].capitalize()
        text += f"{role}: {m['content']}\n"
    text += "Assistant:"
    return text


@torch.no_grad()
def chat_generate(model, messages, max_new_tokens=180):
    prompt = build_chat_prompt(messages)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()


# ------------------------------------------------------------------
# Sample conversations
# ------------------------------------------------------------------

sample_chats = [
    [
        {
            "role": "user",
            "content": "Explain in simple words why the sky is blue."
        }
    ],

    [
        {
            "role": "user",
            "content": "I am building a small language model. What are three ways to reduce memory usage during inference?"
        }
    ],

    [
        {
            "role": "user",
            "content": "Write a short Python function that checks whether a number is prime."
        }
    ],

    [
        {
            "role": "user",
            "content": "What is the capital of Austria and tell me two interesting facts about it."
        }
    ],

    [
        {
            "role": "user",
            "content": "Imagine a small robot living in Vienna. Write a short funny story about its first day."
        }
    ],

    [
        {
            "role": "user",
            "content": "What is the difference between attention and recurrent memory in a language model?"
        },
        {
            "role": "assistant",
            "content": "Attention directly compares tokens, while recurrent memory compresses past information into a state."
        },
        {
            "role": "user",
            "content": "Which one scales better to very long context and why?"
        }
    ],
]


# ------------------------------------------------------------------
# Compare
# ------------------------------------------------------------------

print("\n" + "=" * 120)
print("CHAT COMPARISON")
print("BASE Qwen3.5 vs PDelta3-CLVR")
print("Accepted layers:", accepted_layers)
print("=" * 120)

for i, messages in enumerate(sample_chats, 1):

    print("\n\n" + "#" * 120)
    print(f"CHAT {i}")
    print("#" * 120)

    for m in messages:
        print(f"\n{m['role'].upper()}:")
        print(m["content"])

    base_answer = chat_generate(
        base_model,
        messages,
    )

    new_answer = chat_generate(
        new_model,
        messages,
    )

    print("\n" + "-" * 120)
    print("BASE QWEN3.5:")
    print(base_answer)

    print("\n" + "-" * 120)
    print("PDELTA3-CLVR:")
    print(new_answer)


# ------------------------------------------------------------------
# Optional interactive user prompt
# ------------------------------------------------------------------

print("\n\n" + "=" * 120)
print("INTERACTIVE COMPARISON")
print("Type a question and compare both models.")
print("Type 'exit' to stop.")
print("=" * 120)

while True:

    user_text = input("\nUSER: ").strip()

    if user_text.lower() in {"exit", "quit", "q"}:
        break

    messages = [
        {
            "role": "user",
            "content": user_text,
        }
    ]

    base_answer = chat_generate(
        base_model,
        messages,
    )

    new_answer = chat_generate(
        new_model,
        messages,
    )

    print("\nBASE QWEN3.5:")
    print(base_answer)

    print("\nPDELTA3-CLVR:")
    print(new_answer)

    print("\n" + "-" * 120)

Accepted replacement layers: [3, 7, 11]

Loading original Qwen3.5...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading PDelta3-CLVR model...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


CHAT COMPARISON
BASE Qwen3.5 vs PDelta3-CLVR
Accepted layers: [3, 7, 11]


########################################################################################################################
CHAT 1
########################################################################################################################

USER:
Explain in simple words why the sky is blue.

------------------------------------------------------------------------------------------------------------------------
BASE QWEN3.5:
The sky looks blue because of a special way light behaves when it hits the Earth's atmosphere. Here is the simple explanation:

1.  **The Sun is a Bright Light Source**: The Sun is the only place in the sky that is very bright.
2.  **Light Bounces Around**: When sunlight hits the Earth, it doesn't just go straight down. It bounces off the clouds, the air, and the ground.
3.  **Blue Light is Most Visible**: Among all the colors of light, **blue** is the color that is scattered the mo

KeyboardInterrupt: Interrupted by user